# SciGeoGuard-X — Strong Decomposed LLM Baselines v6

## Obligation-by-obligation scientific verification with 7B-class LLMs + targeted human audit

This notebook follows the v5.2 finding that generic 3B judges were not reliable enough to serve as scientific adjudicators.

v6 changes the role of LLMs:

> **They are baselines, not ground truth.**

Instead of asking a model for one opaque `PROVEN / REFUTED / UNPROVEN` decision, each model must independently evaluate a fixed set of **scientific proof obligations**.

The final LLM verdict is derived deterministically:

- any obligation = `FAIL` → `REFUTED`
- otherwise any obligation = `MISSING_EVIDENCE` → `UNPROVEN`
- otherwise → `PROVEN`

### Strong baselines

- **Baseline A:** `Qwen/Qwen2.5-7B-Instruct`
- **Baseline B:** `mistralai/Mistral-7B-Instruct-v0.3`

Both are run sequentially in 4-bit on a T4 GPU.

### Main outputs

- obligation-level predictions;
- verdict-level metrics;
- per-class recall / precision / F1;
- FSCR;
- false-refutation rate;
- exact obligation agreement;
- model-vs-model disagreement analysis;
- model-vs-SciGeoGuard disagreement analysis;
- targeted **blind human-audit** package.

## 0. Colab runtime

Use:

**Runtime → Change runtime type → T4 GPU**

v6 loads one 7B model at a time in 4-bit NF4. Predictions are checkpointed every few cases so the run can resume if needed.

In [ ]:
RUN_MODELS = True
MAX_CASES = 180
CHECKPOINT_EVERY = 5
MAX_NEW_TOKENS = 900
SEED = 20261002

MODEL_A = "Qwen/Qwen2.5-7B-Instruct"
MODEL_B = "mistralai/Mistral-7B-Instruct-v0.3"

# Human audit is a target minimum. All priority disagreements are always kept.
HUMAN_AUDIT_TARGET = 60

# Baseline health gates
MIN_PARSE_SUCCESS = 0.95
MIN_CALIBRATION_ACCURACY = 0.70
MIN_CALIBRATION_CLASS_RECALL = 0.50
MIN_OBLIGATION_PARSE_RATE = 0.90

from pathlib import Path
from datetime import datetime, timezone
import os, json, math, random, re, gc, shutil, gzip, base64, hashlib
import numpy as np
import pandas as pd

random.seed(SEED)
np.random.seed(SEED)

ROOT = Path("/content") if Path("/content").exists() else Path("/mnt/data")
WORK = ROOT / "scigeoguard_v6"
WORK.mkdir(parents=True, exist_ok=True)

print("SciGeoGuard-X v6")
print("RUN_MODELS =", RUN_MODELS)
print("WORK =", WORK)

# Part I — Embedded blind test set

The 180 workflows are embedded directly from the completed v5.2 experiment.

The LLM-facing table excludes:

- generated expected label;
- SciGeoGuard-X prediction;
- previous model predictions;
- error-family name.

In [ ]:
BLIND_GZIP_B64 = """H4sIAA+Tv2oC/+1d23LbSJJ9769Q6HVNT11w7TfvWOt1tO3uGHm7d6OjAwGREIU1CGAB0LY8Mf++VbiQIFEgCQkUAfJMtGZ6hCJFVmVm5eXkycT76nvfnCiZecmrqZt6jj979S1KvtwH0TfHDzMvzF65Sebfu9Msdf43jcJXaebFxb/+RF/dvvtv7vz7h/ef3jriJdHMmXlx9uAQSl9d//P6eh65wfX1z1fX17Wn19evxC8S7/+WfuLNnP9bumHmZ49yXbgMgo2Hy9DP6g9m3tRP/Sh0Mn/h1R8s3O9O4qVRsMzk44Xi3VLnzk08xxPf50E+vneD1Nt4Pg2i5cxZuOmXjefyvZfh1EsyV2zJxgfNvEUcJW7gZFHgJa5Y5Dwov8hXN1h6jlgxX33sf12/uv5TbFLoFl9FvluSiD8xsUmxR/Wtub72Au+rK79d8bDaGvH5il+kUzco32gaideFy2iZFo+mSVo8uPnt9t3PnBkar05hc8tssvGtao83vlacRFMvTf1w7ogP5QX1Z+kyEdLiOYto5pWH//b2Y/Hnvoo99MXHdGZutlyUH+ndR0aIVSxwp2K/Uj9THbH71fUD9y7wGk/Ex/nqhXL3i7f8exR7SehPl+nV25vyT9+54cxJomDjhWEkPojrpN5CbvQ0rT9rOXE3EKpRHejGC6K71Eu+ikcLP1T/3v2+8U5ZVhzLP//1r1dXG4LwzRWiMCl3UiUL+YJq7zelYdqjOFDy+ggCUT1QScMf724tzfGCwI/TyJ/1KRUPj7PEXQb+9Kr8JMMXi79KExHF1UEu7vzQc6qdK75D7CbuoniJeLn83/zQ7vITLGVr/Q6hfHFpWJuvLm19/tL85KOVjZEf5yfWavK5BpP/dJNPT2byKUw+TP4pTb6QBtuAoR+Iof+yYKWp53VT70XOIpBf2p/lBskhmr1l77eXwOi3Gf2bXyepu4gDL51wptkqbc/krSB1q1y4qfIrPRuhwq/Evxddd5fZQ5T4mZC4r95EPJ0tp9kYNT6NA6Wiyh2tXIXigIpFuYA4XjgrHjLCtAkxJ4R+JuTn/J9/y/+7XO2lmZNmQiXqyw3V8lz3tUL3//Pmw9tf/+vz5D/efHz/4X9yLXVm4kTCXDcXfrpws6nw/QjbsgUtK2ES2kzCzBenE6pv/s1nR3ICYQQGce0ffGmT+qU9rcIzvRmeJd4iyjzxZcP81Ag1lXHa5jIoapui5rs1kX9/wpQxW7Gd3neZO917awsnz00eOyorg6q+tKqK01nJnePG4q72Ziv5a8Ryt1QlGHfu9Is4+kyEccf05oYiGzWfRP9MFT7JBdj3bblgKrlIvPvAmypu+AuQC8YuUi627/3yCtq43Hbf/o1ofxHLi1Ne31GStqUL6CpdwAr3YeNCpGzrXcU7uoe4IQtPnEMZiMw9IR6pPy1eFy2zeJk5a591nV4w6q5KII5rKg43caYP8v4VbgrfclO2l8BFaXNR/oiSYPZ3uVUtLorcSiffy/0Gx828udCxVQ5q0+JowuScn5Nyc/vmar2JxzYof0rbLLeLix9N/OjixxA/pvixxI8sytr6K2HDyV/9W54iwdRVze/8wA+FDpTpBTeZe1lDc4z8Zd5Xf+ZJkY7undCTsIL7KFmUKcKfr7Jk6eUGwdwfuxAdsUs/sQs9ZuzSbhUoQpehhC6F3iFy2emhUkQuiFzUcmEgchlM5MJfMnCxlLWRdOqFwsUREhW4sRBB94srwxiyXSVVroPL0uayTB/8OFXZHn8h9xfVUVRHa3pVFcTqKnafCI0vYw1xmNzOldjek32gyD48P/tAT5Z9oMg+nHP2QYjQ9EeU9ph8ELuwZRDSQLzSEau9qZtm4iZXJSTzRVfVol6NQuVtWhOqf6aKPPlZWIv3H2/+8W5y7wlVSpTgBwlzER+rgXlcXBz6oSYQhhLks21IPr0RluTdbx+v8k0eYdSJfphLBUdvXxSxMCj+VOUBlqajuMvIX1uBYBWe1s3gbixscRuUnW1vb36/+fDrbx9vPn127pfy79QvEAXuLU+gHvs+kNmHc70Pcn3HddDpOmjBfJ5tMNjBGEhdZk1drsUVeZdrEkdBpdLbbl7rWoSAbUq8CpZRfx5P2maUgaAsQXvVRdtPIMib5mIZpstYyFHmzervPpcIfEJIA/veuhomo/3eT1fVxU1jUWmhUAPhxWaPx8v1wmQMoqB0akBKiemVtkBr7XbWOLqd0e28Cug/I6BHtzMILkbd91yW9+khPVQWcIj94BD5iXqoOHw9ABHHAzijBEBEABEVcsEvVS6GCURkWg9IxDhwwyqQ3YFDpMZughaqg6Dl+QQtpk7Az4K8VH8IRGpyq9BfczcGUWPAIAKDeAq1rX6LYtPGvTxd3lXtAS35Zd6t1GQ1S01pLNaJc83NTa2NYLvIpFwHg9BmEDZuadzkuMl33+SFcpV3uSdXrU9R55ZVKEzshYWe58JXWgBTL293e493zuGdP987102wJ0Kne9BpQtp1mlKr0GlGlK1/C7841/Xuigt7u4GouQgK3qbgy/BLGH0LJ34YL7MOlWIodqXY0f29PxXiPjKd/vOvnxhV6lgdhbnw3NBx5/PEm7fhsHcth97txW0eN2TGdXo2oXSlVp7TOPPD4+qFN/PdsBowwQ5svTDQeoHWiwG0XlALrRctrReM785wMx0ZbnD8IcN9thlupu3rqmfsVF31BF31uMs3BOLCuuqNk4HwDXTVX2JXPdPbhwfaaKdBOw00+QTtNJgihPaZ57fP/FKl74yWyWGrXVmVy5rcmFINc4vlBleV5T4H6y/UQu/f+j+4P9xkptL24oksNd65si92u2lajhQrR7MFQg1hAI6epRP7QunRS2fmbsgJZ4CcPB9ywiwLkBNATnqkpNVss7g7rT36S6C/z9dfqjMoMBS4V05pixUKvAfzySgUuIeOLEahwFDgnkZma8Kh6zAyW28dmc3JHkp5C8VutHOh2H3GJPKcduUOtMAdCO5AOASj4A58bWz8x+xEJciZMi8ePzym+dEW7SZf/TUP8XZyvG0p7EKbXfDDmfddZRY+vf39PTLiY7MME/La1LllmAbTTMuwdcpUhoK8tkzONMNitmmLf+heDHuayorXhkgdYD/KD0VLkV7DHDg/vK3bQFs32rqh4gNoAbU41dtbQDVqkUK59yNaKRCtQLQOA9GqfybGPkTr8bMDmAwF5Nu5YFj5AezAhIMdeATswO2pXpADgxx4ZNPoQQ4McmDIxbDJgSl7OXJgbjSTUPmZzRNfbmm6cLPpg4I5pbkInkpr4cmbjdHgXF9/YG/KSGjobkptk4d660TZg7r6+CJyYFyKHIR+MjA5aGYgxWkn+ZjQVeFpw7BLw9Iw7rn4OI3GGu+7fDfxguK5Yh/EisCf+pnjBv48XOTB0EbJ0VSWHDeRjOtSxDYXh3Id7oK2u2D64MfKQoS/kPuLKgSqEJ1AxbqtFVq8pymAoamnB0yxxXSGKiL0tz/9NQjhhQLbXZE/DMgfIH+g0DXkjxLow14a2KORZk5FvOsycKX/7rkK1a0/hrrurvjEUfA4r6zaC1d8KFT19HfvEdOhGu14CVOCSxiXMDR7A35rMWZqlm4apk1Namtq+K2p25xoBje5WG3r2otf0+zA0gdB6QOlD5Q+Rlv6YCh9oPShLH1ofD9Kj1Kg9PpB6bFjovQMm6PlcugovVLzANPbOatdB0wPMD3IxUXA9OaeEI+0GqawKzOlNaPVbfFTtBQ0JRSOitpRieIaFe5JTM7r58Qpn39FvNq/hak2syEBu0yN1FZ9z0QjBpInTDQ6jZ7e3L65Wm/iWY8dPCblU7cJR5oCZ1/u7w8viRRZ5tpTGINdswjR5X2qzMJqWnW1zSe8se/9ICg16hBtzz9MXhiyxX/kb3I19NNVrTd4LMCRm2psHuSFU3jhl+mFJ+435+0nOOLDccSt1llS6snAmCWFWVJg1DjyLKkp3DAMk3r+MKkviyo5au+v43LUccG2gjou2FZeilUDZVyUcVVyYaGMO6QyLn/JKq6uaA26X0r6uXrmWMSmpspZOToxKD1fYtA8aAUvaDdeUCEP+v5J92djkDpwREpdpgdQPBIEHQg6EHQg6EDQgaDjtHJhIOgYTNBhvBzDo852EwNpHMRAPQwb1TUQA4GcoKdho+aE2R2GjZqtw0b1AxrcNAMxygga3DBUEB1u59LJRBClIEpRyAWI6C82StF2RyncQJTy/CjFsG1EKYhSnj0Ekeo6bR+CyBkrOIl1XcmmVvvmTtlAoOhclfjI/E+6wVWloOeg5EIB9P6V/MH94SZKqqXiiexSuHPlAOvt6ebgVHtpVRf7wjk7qrL/+ddPutGOfGdAvj8d+c5Phnznx0G+fwbyHcj3AwxsLg220acsAP7+DPj7L1XsZB7AYqYhydtPkpcSME8jx4scL3K8yPEix4scbyf4u3V4QqjhsSAhhITQ2SaEiKUdPSFkN3tPlmG6jMVhSBbl2mnMJSaFkG0N3LEaoUN740maqcnPK1EWsiRM27Ya9usIQPGGcOE/jY1MGunQq4qoLXRk5RHv4SNbU58bik60Gh2eeKGQ2ng9bG7bFrSuhSVoswQrjseWHMJLEBcijXAJ/IXSWnhVm2iLwbA78RcaBzS7MR05RuQY4SWg2Q2E+EgxnlYuOFKMl8mwYbBt5vU0EC6OU1FnKNyU1aKj82vo58uv8f7jzT/egWCjI8GGphSITXtUvdGorqQKLGWcDCxlgCZ0bKQqG7dEdXfVDd9ubExh//nuRgJqoZHg+Y0EFjcpGgmQ0+6t3Zl3aHe2WtudDa2lvuxnzqqoWR/tzBsVLuVK2IQ2mzDz01qEuWkMNp8diQscVmAQGNiDEaxESeBs6AfN1WCYq3GZczUwjX1IQzUMYw+lEIGP/Xwfm5u2BR8bt+shzbpTL/Qqr0r8qyNBBIEbO/eJUNyylkte65ZlFAps7h5P2aAfxnhKjKfEeMoxjafUusE7rAPgHeAyBk8YbnP0kL0cay0FwAMAD7VcAOAxHIAHZS9HFGbYe4jCKHIPz889ME5M5B6Qe+gv96CZhp4rsKloPBFyuQzcxJF2Q5F9qD+G+u4OMOIoeJxXVu5oAYaGAGOwKnvEy9ek+8CVDZ7OFwNXUoArAa7cFgh+1uDKs2OiA7hy2OBKk3UgdKXg7wB/x8UQuhYycEz+DpMrOvbXyeV2xWsuQhDVpna3bLLCGqnZOgCXOm+41J6KTR8ZWHmTanvb1Cy0qSGSGkabmqFE559RJGWfLJKyEUldZCSl77P/lMD+w/6Pp035+vrTm9s3V+9++3iVb/IIYRZnextgwsPAbwNj322gnY60guI2wG1QFwh27tEAJvzA/r+s/d/Tj8PRj9NDPw5YV9GPczqCZtapIce09nmE5GT5AUbgEcIj3BQIYoPGbEw0ZvAIB+4R2rs9Qo3CI0SH9ik8wuPb8VH6gNPlXcVO2+IAGp0cQIvsMQAcBgAhIULC0ZoD3s0cUCXydrEMMr/o3cxPrc5tuD2xp3UtLEUrGvCU/fiv0ZGPjvxRSYaNnvzDW+pr8+BbWuql2We7u92ZiW73HrrdGbrd0TrbK9OeOOtCgRVNG2ksNkqcRv7XxDG6X9ycdM/e0mXlOih0m0JvKCkUGYq8W5EL5SpV2ZOrarGZbbNCYWIvLOKzXPjK3J3GtUK5taZyi7cRyzw3dNzUcafTpQi7qhmq2ymb1rVQ8p31OZWKx4n43nEuuuLLyZ1tVuj+9tCxLrMrfUP2zFQWlwD0/Nh6vtp9dz5PvPnq1A9P1Ej1K6Uivxuc2bK0BPVTFEoVuNuqKE+jrrOFTdD3zJ8BN3YPHjvVbQMXPQzAcy963Sry8OqLnmpmGYYbzYv+fimrvvWErcKFzymXgNB+us7n6AvAcTrAccwJpwe165yNDegAzZC6bCorKfHDY5ofQ6H/X/2octvpttvethQ3d5sW+8K2flep76e3v78He8nYLuwJec0NyyDM1HRqmpxTW1MpL3ltUmLrFrGp8Hstzvle7ro0lXOkNoTqAH++/Fi0FOo1jMo6gNe+0VoFXvsn8trzY/Lat8fiHHo9FJKUAq8AVvud4+kJauhgtVfLBVjth8Nqz1+O1N6yD8wvaMgvIL9w8gZwekC7z8XmF2wFv/0yTJexONvMm9UPdy5nYTf5GnesRhjSrsjpypNUEzcKCYzF53g8nlehIxAZgvNwxCZeyop64B7Y9oor8iebKkhb1z0D4oVCauN1rnF7wnXrWliCNkuwaoRBe9d4LMEomzykufAqt7utzUPv0uZhqwnWV4JUQAErzCGh266DeiFsRavX4E2jZKbGDMoWGSFLPlAEQBGsDqjUdRkWOl44qwdGxoRSJTNWJmyEkwr7kDVaJraXF0aAd/IZiAGfoT+fAR2h8BlOPbubdfMZtANqnJjd3VONk55odjesA2Z3j6qapaPKiSqnUi4YQ5VzMFVO4+WqnLbesTLSQF6iMjLSygggWudeGTG7FkaMDqMkCUZJYpQkLEJPgyTtrh0QxEQHBDogoHQbHRCEawaxKLcNRk2dGLa6A8JiuqVzWzMNapuWrb90B4RtdaooNjgKUFFERRE362kqiuaE8H4qigp88dvbj5J25O3njwoHe/0Qmt6m6SU3PEYFQeGfwvNezIGmhBweBzfQgYiDEQef520t9oVZ9NiRMCW0m3dM4B3DO4Z3PBTv2OzFO6aEHdh+R9B+h/a7IbTfGWi/a2m/o4TvHpzCLAxOweQkTE4a1/TMbsOTKNEOvNEt3Oi40Qdwo1OKG731Rtf3zcKlxqlm4UqYK2bhQpc3BMLCLNwxzcK9xSzcQc/CpUSNFczHJ6TLhfPNFxfjMltTtxOiq0YtqNYi2msz/8VEhcl6lMLgBy5QZG6HP3BBdiTP1+LSMnWBdpu6QInZeT6mhfmYmI+J+ZiYj/kykqEps/aYj/m0+ZhGafYV/N4zL84eFKMxa09h0feA2ib2yYI8G0HeEAz4N+GhJZNyJ1WykC+o9n5TGqYjzv9cX//x7tbSHC8I/DiN/FmfUvHwOEvcZeBPr8pPMkrrfeeHnlPt3AHmu9G2HMoXl4Z1d/xfWP7tRmJK7Hazr8HsP93s05OZfQqzP3azP2qrfwRhgLV/hrX/pbL1VA2Q35y+vRqi3YDIK9fhImi7CKYPwutR6b2/kPuLsZoA6Nb0qjL4dRW7T4Q6l4AccZiWWaox3TMX18Rc3OfPxTUswjAXFwrcnwIzapUKrKam3jyRen1l+yZuWQm9HmiD6Vn44OdST8nDrkuKx46YhTunSkr36Cy35VxdLS8bjdcbpuh5ai6CBW+z4MvwSxh9Cyd+GC+zDnYcDtmYcJJlLzFV9ByksTfN8k0XH14x6mvjOfSovW14NkYS3evrD+xNeQkPPaKpbfJQfaAoe1D7QJCDPuUg/eYn1BiYKDTj29Jw1oj1NhwfaVsaOMRcgpxGztn7Lt9NvKB4vt6K0C/lTawI/KmfOW7gz8NFzvJf7zejegc2UwssLmBxuRQWF0qN43texu6Wb81EyzdavtHyPa6Wb61byzc1WzFBnAETBCgoMEHABAETdCYIUGrthhNogBP0ACegBjcAJ0ClqS/CPtoTYR+1u06S0jFJ6jwmSekwB2c+SYrqpNMkKcpI195vStD7jd5v9H6j9/tlJIN/Jhy93331flNW2n3a9AKFzVoGrqzxea6CBqj+GKZ997jzOAoe51F45HHnGgaGDlZbjzgLmDLWmq6nSNejhRfpeqTrT2jDkafvJ0/PeKfuXYLuXXTvwm8bTPOfbpdqrDW9tcRbRFkRyuW4Rs1Wum2by6C7u4Mu+fdbkFY9BV2Gzfm5waxGq7vicFZi57ix0FtvtoY1befVTplxHV5WTTKtI9966nzrIOWCMWRba3zr9att30zarQDgoHQtXUUQTJGuZdrWuz4lrTT3hHik/vSAxJLezg1nI7GExNKpRl0jsQQcKPJLvXLDMaPb+GYD45sxvhkx6iDQoMaE8n7QoEw9+yN+eEzz48yF2vnqR0HbHOe2pbAEbZag1na9aQc+vf39Pdpux2YNJuS1pbIG9DXfCxJIU3mHb0jNAbFd+XdpKbU1ciLWPtNBowjg0MiHAA4BHAK48wjg7I6kogSkoiAVBakoSEWfKgu2AUnok0qUkyYIv0b04shxn0m8Cr0bCK/WtTDhbSZ8xV4ESp7xKO0oeXqEJE1/ROnOdk2jG00Pp10AoUQDIBSAUKTohgII1YhWqjHbDwglJgCh/QBC6WkAoRS6C0DoeIB/BIBQAELVcgFA6IAAocaL4kE574YR0oERAkYIbstAMEIK0M+TMEJc200R3uAcAEU4KMJBET7s1GNHhnCud2WN1MAaeR6skShXnD1rZOn2HU4byY1OlQgDlQhUImADhlKJ4LZRqrF5QCWCoRIxAmoKwA1GUIkob1CUIlCKQCmis1wYlyoXZ8xNcSjlKbea+YfyfH54SaQYD117Cu+kPb+ghrPL34tP00Qwg+SyR2MiW/nStft8Upty7wdBqVGH6HD+YVb/L1dBP121DQePxbSorRyivbuOwFFHQB3hROp8c/vmar2JADQ/saqgd6sqaIoGiESIn7Pw3NBxU8edTpeL5Yp7YLum0LoWlqLNUrz/ePOPdyoLESfie8f5hSa+nNzZ5uX/t4eO13+7sRCSTXbbC/JaR17i2Nf+avfd+Tzx5qtTP9wsyMaj+VpcCtvizJaJ2zhKoVmBu62P8kjqiltaBro/M0mRmRwDRnqXDUBmEhhpkOYiMTluuTCRmDw3jPSheUmN7c5pNCDRyGkgp4GcxqjgUR2hkhpvJjXybZ8n/qzOx7NdsWgugnFo75SYjdEHub7+wN6UgfLQI5faJg/VEY2yBzU6FnLQpxyEfjIwOWhC48RpJzkNzopbdeMWkIalcRPk4uM0mNa87/LdxAuK54p9ECsCf+pnjhv483CR50c2rgCtlYaTc9BwgoYTNJyg4QQN55hpOBerDED7zBxNh63HzBwMY4ath60/D1tvdCLsbLTOgLDzGYSd/GRJYI469QXkgos2uh15YLtjHtg8hJ99GX4Jo2+hIh2sXghLAXZ2sLODnR2ycHx+ds3qSHpCbJCenAfpCVy+cyc9MTpSnmh2Ny5EBi5EcCHCBgyCC9HsjQtRJ1150BhcgvNwCXSYg7PnQevoE+i0aQ1WUBS5pwo/YOM5NB4APwD8APADwG9cAD+dtYI+GAHo4+mgD34y0AcHwG/soI8vo07h//Hu1tIcLwj8OI38GdAfA0F/1Lr9dN6J9piC9hi0xwj5h0J7TEol1vYH7QRBO4J2BO0I2octB96s8h8QuB8YuOv7eaUYeKV64pXip5m9C/AGGO9HRSBEQCwFYimlXGD47oCIpSh70em7urHNLZUG/sxz7qPEm7pp5hCuKdil8kVX1aJe/ZRKNC0JkKEK0TwfitzJvSeUNfHAjr/Li9kQCGI3BWLbIn16c/vm6t1vH69qPMSjuqLOlokC3ckd76RYGBR/qspMlqajaIUjf21dGtVdVjeDu6sR5XVg7r0OyKmuA0mNiusA18FaIDSlQLz0+OYjXgDGyS4AAxfAZV4A1m6uWWqCa/b5XLP0ZDQD46XDP74hP192WaMbq4DerQ2N6mhDQxsaqhmDaEMzJkzrpw3NIO34Ywr8MQhG4dWDdK4zg4htAHQ8GNDxytTTVlNPNZh6mHq0mqDVBLZ+xLb+l5WpZypkurBGq3B93VvSbCtXrMMl0HYJbDSPbIX18naQOoYGE8T2G8pVmnxPrqqdosXMQmFiEe/nSbxc+EosqqGV6XtD3TyWz7BOlwvnmy/2c5mtR+c2lbx1LRS9TdGLsdaT9YBiTL2Gxj9/6nW6XPQ679rYmBjkRc4ikHvgz0pTwLeZJbeXwAK0WYCbXyflTT4xuanjyocB6K+nVNdL/dWVl7tQ88wvMMf55tbGPzaSN61rodptqn3SPpLXA+wYkLBsdJKcupNkkJLB0EvSqRWkludRtIIwrbT7Rlc+UAI+0PPgA6UEntyZE4LqpSk/nBHUMPf3l2sU/eX99JfTY/aXj6JZ9OIbzAtkJPrLERWgv7yzXPBLlYth9pfzF20vN6xWYBExASzC4GIAiwAswuTi85hcbNi7ewc1Db2D6B08TUR6c/vmar2JmE38lNnErFsXoUl2ww0axUnADZ4AN9A50wA3QJL6uQhDw9KMdoQhtQy71GqqBCHUvrrEFBRJaLqdhJZOd/433eCq0tBz0HKhAXr/Wv7g/nATJdl18UTWn+5cWWTYLkGJnREbEsqtEaqfXhpFzAl0XewL17Wjavuffwnt2zETyEA+BY1ayKcgn4JGrfOYBGTy3fmUxuwf5FOekE9hJ8unMORTzjmfIpE/XsWG2Q8xk6ntR/5wHciffpA/7DSTJRgiLEyWwGQJIH8wWQLIn9Eif0y92bQgbt5l4MqZW54rwpftdtP6Yzgoux2UOAoe51Ve6qWhyZh9NQCj8hT1jUVYW7Uj7FReY3fplqB020Pp1tJ1jtItVLkf4ldtQnkH4le9nfjVNJUlXiGkstLoCIPi5hsZRGmqyEIq18EgtJM/b8UH6l7Di2srvqDkQ6Uxgfsokwd5LPn3D4U6blqKWRLFDUU8xAtYyeQs+hammfAdFltdhabVmV2Cg10C7BLoIwO7xItJBkc+qS92CV6affWsj00+ohUlaAPRp1wHc99m7qcPfqwkBPUXcn8R8CHg60QNxqlRqLFF9g1uJAYGN2Jw4zAGNxrKRMXZzvHlJwOXcoBLL3GM4/8DvXIy2RB6AwA="""
REFERENCE_GZIP_B64 = """H4sIAA+Tv2oC/61a247bNhB977e4AC+SZT+2sNMG2EuQeIP2iWAkWhZWJlVK2jT5+lLuer2+SBqO+LQGFsPLXM45M1QqayWKbLa15qfSorIqK9KmMHqWK62sbFQm1L+VSt2P2V426U7V4vSvUn5T5UyLbaGzQuf1L1/++IuL3+8+PqzEtjQmE5mqmp0glM4+rz88bdart78b26oZ7bXg0W0L9t5CGbEvxYssi0x2pxYkWg5u9Of6bvX4tPn1w2/3H+/+Fq0uGpEVe6Xrznpf1Ic7CkLY7NPnx6/rh+OfwyLk+rRW7U2jRN2toHN30QRw0VLqLDUvyop0J3WunBkH++dyRxL73LhOXfBEt3cpK1Eq+Sy7/ckSd2wKNKtdhJTYGqtSWTduPz4Y3dX66/ru8dP9+mEjtm3TWpek2lnvX6M8GJ73xqlL09zYIpWlW6FRtjLlcQ0OXKPVdVtVxnYJb1VtyvawQC6LbhUCTJTXtI7wgaYLgOl1RdAYFdqIgSNUV24r5+C6Kl05nXJq3DU3TjsUlYtcduV68Exl3cm11Gm3KYXbv0+OvZJayDy3Kj/l2JDfRlN0jnI7iwGwd6uiGAMYniPsctDiCipTZRuX882PN8d3sOEfYs5QecwJyozR0SPewLUFOPmH0WHhg83V7kd9SEZ7OMVLccKqJRCr+kpxDnLCVU5RcE5dsRIHV883t7PIbZGd828AUotxGcMCBQ0OIM60LaUV0irpe/n+A1ACzJqbESB4oqLgizvoTtX/YFKqF1WOJk4vbsKdrY2LtBQ/lTUjJDVyUm9N68rw6eF1n7cfUK9yOo2PErzCxCHvBKkTzVHin8/9xP8tRuN+eo4wNETSaDL/uiVCaNgogJqOkA2TEzwoaopRGeKIHSwP+5tDjscMhhBMEUFB4oSKZ7iK94CoC8ZLUEnQV+/co94pNPdL0zp+lPXzmTGu/WWLwYP3G1KCclQE7SrOM4gn2PaegIjuRtdJUQcdoZrLvrEtm6LLdWNF4+r7fWlHiPpkCTjt+/Q5VN93Y7fXXrUWMk1bd5fTPAM1HoAKoJuiYgmH0wGdypHcEcXTJFEY9oSD3gB/Ugb3ZKP27jBvWXTsezrdPZ3H53jRRsI0zZRPV3Fj+ANv4pLpcfHohVdf7rvKXm3uR1oTkDoMkFMj2n+0xAhi+sIW04bDCxRHUi8xcQDiut2L70WzM60Tiq097h97jDAHqGjh21728VD/CJBFAcYsFDfPG3kpudjbbdocKtSZO69jJ0U3R8ZjYHFO4CptujqxplSj4A/CqgWiRCLvUCOnrhFc2wyDepiioATbYcR+fT1lPs15X2XgZ2jR0u+8zOvdsxft5yH0nMfo8PgqRsNgAQkixUiI2XOEH7clQUIZY949WCAJF4JXJkhRAr/H+TSYoiYtHD58HmrgsCM0ylBNM4mnPdFQP4TinpPVKA7QzXgo4AtUafWzNt+1x4RouEtcBtDiLBAXM/Br4rnYYZ6cRAJwKNj9l4fFcy9jyLEXj7CGBFXBNPFxcO/YwFMaMe93r5GXDsiAzEvlDzVnyI8PloEGjCPMCK/hCdLS/90ywbQomFHq4IMWaObhyUpsjuNOiuRqHoNL4aKBwXxqQzzc+U/rDDtXqkYeJFFp6trvg6qhvPf4rquvm0J/bNgrJP8DBIZ2cf8qAAA="""
V52_GZIP_B64 = """..."""

def decode_csv(payload):
    from io import BytesIO
    raw = gzip.decompress(base64.b64decode(payload.encode()))
    return pd.read_csv(BytesIO(raw))

blind_cases = decode_csv(BLIND_GZIP_B64).head(MAX_CASES).copy()
frozen_reference = decode_csv(REFERENCE_GZIP_B64).copy()
v52_history = decode_csv(V52_GZIP_B64).copy()

blind_cases = blind_cases[[
    "review_order",
    "case_id",
    "workflow_intent",
    "artifacts_json",
    "steps_json"
]].copy()

assert len(blind_cases) == blind_cases.case_id.nunique()
assert "generated_expected" not in blind_cases.columns
assert "frozen_prediction" not in blind_cases.columns
assert "family" not in blind_cases.columns

print("Blind cases:", len(blind_cases))
display(blind_cases.head(3))

# Part II — Install current model runtime

Both selected models have native Transformers implementations. No repository custom model code is executed.

In [ ]:
if RUN_MODELS:
    import sys, subprocess

    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q",
        "transformers>=4.53,<5",
        "accelerate>=1.0",
        "bitsandbytes>=0.45",
        "huggingface_hub>=0.26",
        "sentencepiece"
    ])

    import torch
    import transformers

    if not torch.cuda.is_available():
        raise RuntimeError(
            "A CUDA GPU is required. In Colab select Runtime -> Change runtime type -> T4 GPU."
        )

    print("transformers:", transformers.__version__)
    print("torch:", torch.__version__)
    print("GPU:", torch.cuda.get_device_name(0))
    print(
        "GPU RAM:",
        round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2),
        "GB"
    )
else:
    print("RUN_MODELS=False: model runtime skipped.")

# Part III — Record exact model revisions

In [ ]:
MODEL_REGISTRY = {
    "strong_A": {"model_id": MODEL_A, "trust_remote_code": False},
    "strong_B": {"model_id": MODEL_B, "trust_remote_code": False},
}

if RUN_MODELS:
    from huggingface_hub import model_info

    for key, rec in MODEL_REGISTRY.items():
        try:
            info = model_info(rec["model_id"])
            rec["revision_sha"] = info.sha
        except Exception as e:
            rec["revision_sha"] = None
            rec["revision_error"] = repr(e)

(WORK/"model_registry.json").write_text(
    json.dumps(MODEL_REGISTRY, indent=2),
    encoding="utf-8"
)

MODEL_REGISTRY

# Part IV — Scientific proof obligations

Every workflow is evaluated against the same obligation vocabulary.

Each obligation must be one of:

- `PASS`
- `FAIL`
- `MISSING_EVIDENCE`
- `NOT_APPLICABLE`

The model may not create new obligation names.

In [ ]:
OBLIGATIONS = {
    "provenance": """
Check whether provenance required to scientifically certify the supplied artifacts is available.
Missing provenance can be MISSING_EVIDENCE.
""",
    "categorical_semantics": """
Check categorical/nominal data operations. Continuous interpolation or arithmetic that destroys class identity
is FAIL. Label-preserving operations can PASS. If no categorical operation is involved: NOT_APPLICABLE.
""",
    "spatial_support_resolution": """
Check native resolution, spatial support, grid alignment, and unsupported claims of finer information.
Interpolation alone cannot create new physical information.
""",
    "units_quantity": """
Check dimensional consistency and quantity semantics, including rate versus accumulation and invalid unit conversion.
""",
    "crs_geometry_area": """
Check CRS/geodesy implications for geometry and area. Metric planar area directly in angular coordinates is FAIL.
""",
    "spectral_semantics": """
Check spectral band roles and band-grid alignment when a spectral index or multispectral operation is performed.
""",
    "processing_quality": """
Check processing level, cloud/quality masks, quality metadata, and other required product-quality semantics.
""",
    "vertical_semantics": """
Check DSM versus DTM requirements and vertical datum compatibility when vertical quantities are combined.
""",
    "temporal_causality": """
Check acquisition/availability time, future information leakage, and multisensor temporal compatibility.
""",
    "evaluation_independence": """
Check train/test spatial, temporal, or scene dependence when the workflow contains an evaluation split.
""",
    "nodata_physical_domain": """
Check NoData/missing-value semantics and explicitly required physical value ranges.
""",
    "uncertainty_fitness": """
Check uncertainty only when the scientific intent actually imposes an uncertainty requirement.
Optional null thresholds do not themselves create missing evidence.
"""
}

OBLIGATION_NAMES = list(OBLIGATIONS)
OBLIGATION_STATUSES = {
    "PASS",
    "FAIL",
    "MISSING_EVIDENCE",
    "NOT_APPLICABLE"
}

pd.DataFrame([
    {"obligation":k, "definition":v.strip()}
    for k,v in OBLIGATIONS.items()
])

# Part V — Deterministic verdict aggregation

The LLM does **not** decide the final label directly.

The notebook derives it from the obligation statuses.

In [ ]:
def aggregate_obligations(obligations):
    statuses = [
        str(obligations.get(name,{}).get("status","")).upper()
        for name in OBLIGATION_NAMES
    ]

    if any(s == "FAIL" for s in statuses):
        return "REFUTED"

    if any(s == "MISSING_EVIDENCE" for s in statuses):
        return "UNPROVEN"

    if all(s in {"PASS","NOT_APPLICABLE"} for s in statuses):
        return "PROVEN"

    return "UNPARSED"

# Sanity checks
assert aggregate_obligations({
    n: {"status":"NOT_APPLICABLE"} for n in OBLIGATION_NAMES
}) == "PROVEN"

x = {n:{"status":"NOT_APPLICABLE"} for n in OBLIGATION_NAMES}
x["crs_geometry_area"]={"status":"FAIL"}
assert aggregate_obligations(x) == "REFUTED"

x = {n:{"status":"NOT_APPLICABLE"} for n in OBLIGATION_NAMES}
x["vertical_semantics"]={"status":"MISSING_EVIDENCE"}
assert aggregate_obligations(x) == "UNPROVEN"

print("Aggregation tests passed.")

# Part VI — Decomposed LLM rubric

Important schema rule:

> An optional intent field set to `null` means **no extra constraint was declared**.

It must not automatically become `MISSING_EVIDENCE`.

The model must separately evaluate every fixed obligation.

In [ ]:
SYSTEM_PROMPT = f"""
You are an independent scientific reviewer of Earth-observation and geospatial workflows.

You must evaluate a workflow against a FIXED set of scientific proof obligations.

For each obligation output exactly one status:
PASS
FAIL
MISSING_EVIDENCE
NOT_APPLICABLE

Interpretation:
PASS = the obligation is applicable and the supplied information satisfies it.
FAIL = the supplied workflow contains a definite scientific contradiction for this obligation.
MISSING_EVIDENCE = the obligation is applicable and necessary evidence cannot be established.
NOT_APPLICABLE = this obligation is not required for this workflow.

CRITICAL RULES

1. A definite contradiction takes precedence over uncertainty.
2. Do not invent missing constraints.
3. Optional intent fields with null values mean "no additional constraint was declared".
4. required_unit=null does NOT mean unit evidence is missing.
5. max_uncertainty=null does NOT mean uncertainty evidence is missing.
6. temporal_tolerance_h=null does NOT by itself mean temporal evidence is missing.
7. Missing artifact metadata matters only when an actual operation scientifically requires it,
   except provenance, which can be required for scientific certification.
8. Judge only the supplied workflow; do not infer a hidden expected label.

FIXED OBLIGATIONS
{json.dumps({k:v.strip() for k,v in OBLIGATIONS.items()}, indent=2)}

Return ONE JSON object only with this exact top-level structure:

{{
  "obligations": {{
    "provenance": {{"status":"PASS|FAIL|MISSING_EVIDENCE|NOT_APPLICABLE","reason":"brief"}},
    "categorical_semantics": {{"status":"PASS|FAIL|MISSING_EVIDENCE|NOT_APPLICABLE","reason":"brief"}},
    "spatial_support_resolution": {{"status":"PASS|FAIL|MISSING_EVIDENCE|NOT_APPLICABLE","reason":"brief"}},
    "units_quantity": {{"status":"PASS|FAIL|MISSING_EVIDENCE|NOT_APPLICABLE","reason":"brief"}},
    "crs_geometry_area": {{"status":"PASS|FAIL|MISSING_EVIDENCE|NOT_APPLICABLE","reason":"brief"}},
    "spectral_semantics": {{"status":"PASS|FAIL|MISSING_EVIDENCE|NOT_APPLICABLE","reason":"brief"}},
    "processing_quality": {{"status":"PASS|FAIL|MISSING_EVIDENCE|NOT_APPLICABLE","reason":"brief"}},
    "vertical_semantics": {{"status":"PASS|FAIL|MISSING_EVIDENCE|NOT_APPLICABLE","reason":"brief"}},
    "temporal_causality": {{"status":"PASS|FAIL|MISSING_EVIDENCE|NOT_APPLICABLE","reason":"brief"}},
    "evaluation_independence": {{"status":"PASS|FAIL|MISSING_EVIDENCE|NOT_APPLICABLE","reason":"brief"}},
    "nodata_physical_domain": {{"status":"PASS|FAIL|MISSING_EVIDENCE|NOT_APPLICABLE","reason":"brief"}},
    "uncertainty_fitness": {{"status":"PASS|FAIL|MISSING_EVIDENCE|NOT_APPLICABLE","reason":"brief"}}
  }},
  "summary":"brief scientific summary",
  "confidence":1
}}

confidence must be an integer from 1 to 5.
Do NOT output a final verdict. The notebook derives it from the obligation statuses.
""".strip()

def make_case_prompt(row):
    return f"""
Scientific intent:
{row['workflow_intent']}

Artifacts:
{row['artifacts_json']}

Workflow operations:
{row['steps_json']}

Evaluate every fixed proof obligation.
""".strip()

print(SYSTEM_PROMPT[:2200])

# Part VII — Balanced decomposed demonstrations

These teaching examples are not part of the 180 blind cases and are not reused as held-out calibration items.

In [ ]:
def blank_obligations(default="NOT_APPLICABLE"):
    return {
        n: {"status":default, "reason":"not required"}
        for n in OBLIGATION_NAMES
    }

FEWSHOT = []

# PROVEN categorical nearest-neighbor example
o = blank_obligations()
o["provenance"] = {"status":"PASS","reason":"authoritative source is supplied"}
o["categorical_semantics"] = {"status":"PASS","reason":"nearest-neighbor preserves category identity"}
o["spatial_support_resolution"] = {"status":"PASS","reason":"coarsening does not claim new spatial information"}
FEWSHOT.append({
    "user": """
Scientific intent:
{"goal":"coarsen nominal land-cover data","required_unit":null}
Artifacts:
[{"name":"landcover","quantity":"land_cover","scale":"categorical","resolution_m":10,"provenance":"official"}]
Workflow operations:
[{"op":"resample","params":{"artifact":0,"method":"nearest","target_resolution_m":30,"evidence_of_new_information":true}}]
""".strip(),
    "assistant": {"obligations":o,"summary":"label-preserving categorical coarsening","confidence":5}
})

# REFUTED rainfall mean-as-accumulation
o = blank_obligations()
o["provenance"] = {"status":"PASS","reason":"source is supplied"}
o["units_quantity"] = {"status":"FAIL","reason":"a rate is averaged but declared as accumulated depth"}
FEWSHOT.append({
    "user": """
Scientific intent:
{"goal":"daily rainfall accumulation"}
Artifacts:
[{"name":"rain","quantity":"precipitation_rate","unit":"mm/h","temporal_resolution_h":0.5,"provenance":"official"}]
Workflow operations:
[{"op":"temporal_aggregate","params":{"artifact":0,"method":"mean","sample_duration_h":0.5,"declared_quantity":"accumulation"}}]
""".strip(),
    "assistant": {"obligations":o,"summary":"rate/accumulation semantics are violated","confidence":5}
})

# UNPROVEN vertical datum
o = blank_obligations()
o["provenance"] = {"status":"PASS","reason":"sources are supplied"}
o["vertical_semantics"] = {"status":"MISSING_EVIDENCE","reason":"second vertical datum is absent"}
FEWSHOT.append({
    "user": """
Scientific intent:
{"goal":"combine elevation and water level"}
Artifacts:
[{"name":"dem","quantity":"elevation","unit":"m","vertical_datum":"EGM2008","provenance":"DEM"},
 {"name":"water","quantity":"water_level","unit":"m","vertical_datum":null,"provenance":"gauge"}]
Workflow operations:
[{"op":"combine_vertical","params":{"a":0,"b":1}}]
""".strip(),
    "assistant": {"obligations":o,"summary":"datum compatibility cannot be established","confidence":5}
})

# REFUTED temporal leakage
o = blank_obligations()
o["provenance"] = {"status":"PASS","reason":"source is supplied"}
o["temporal_causality"] = {"status":"FAIL","reason":"feature is available after decision time"}
FEWSHOT.append({
    "user": """
Scientific intent:
{"goal":"forecast flood","decision_time":"2024-08-01T12:00:00+00:00"}
Artifacts:
[{"name":"rain","quantity":"rainfall","unit":"mm","available_time":"2024-08-01T18:00:00+00:00","provenance":"official"}]
Workflow operations:
[{"op":"predict","params":{"features":[0]}}]
""".strip(),
    "assistant": {"obligations":o,"summary":"future information leakage","confidence":5}
})

print("Few-shot demonstrations:", len(FEWSHOT))

# Part VIII — Held-out calibration set

The calibration cases are separate from:

- few-shot demonstrations;
- 180 blind workflows.

The models are still only baselines; calibration is a health check, not test-set training.

In [ ]:
CALIBRATION = [
    # PROVEN
    {
        "cal_id":"P1",
        "workflow_intent":'{"goal":"calculate projected area"}',
        "artifacts_json":'[{"name":"mask","quantity":"extent","scale":"binary","crs":"EPSG:6933","provenance":"official"}]',
        "steps_json":'[{"op":"area","params":{"artifact":0,"method":"planar","output_unit":"km2"}}]',
        "truth":"PROVEN"
    },
    {
        "cal_id":"P2",
        "workflow_intent":'{"goal":"combine vertical quantities"}',
        "artifacts_json":'[{"name":"a","quantity":"elevation","unit":"m","vertical_datum":"EGM2008","provenance":"A"},{"name":"b","quantity":"water_level","unit":"m","vertical_datum":"EGM2008","provenance":"B"}]',
        "steps_json":'[{"op":"combine_vertical","params":{"a":0,"b":1}}]',
        "truth":"PROVEN"
    },
    {
        "cal_id":"P3",
        "workflow_intent":'{"goal":"chronological evaluation split"}',
        "artifacts_json":'[{"name":"samples","quantity":"records","provenance":"official"}]',
        "steps_json":'[{"op":"split","params":{"mode":"temporal","train_end":"2024-01-01T00:00:00+00:00","test_start":"2024-02-01T00:00:00+00:00"}}]',
        "truth":"PROVEN"
    },

    # REFUTED
    {
        "cal_id":"R1",
        "workflow_intent":'{"goal":"calculate area"}',
        "artifacts_json":'[{"name":"mask","quantity":"extent","scale":"binary","crs":"EPSG:4326","provenance":"official"}]',
        "steps_json":'[{"op":"area","params":{"artifact":0,"method":"planar","output_unit":"km2"}}]',
        "truth":"REFUTED"
    },
    {
        "cal_id":"R2",
        "workflow_intent":'{"goal":"resample nominal classes"}',
        "artifacts_json":'[{"name":"lc","quantity":"land_cover","scale":"categorical","resolution_m":10,"provenance":"official"}]',
        "steps_json":'[{"op":"resample","params":{"artifact":0,"method":"bilinear","target_resolution_m":30,"evidence_of_new_information":true}}]',
        "truth":"REFUTED"
    },
    {
        "cal_id":"R3",
        "workflow_intent":'{"goal":"independent spatial evaluation"}',
        "artifacts_json":'[{"name":"samples","quantity":"samples","provenance":"official"}]',
        "steps_json":'[{"op":"split","params":{"mode":"spatial","separation_m":200,"dependence_range_m":1500}}]',
        "truth":"REFUTED"
    },

    # UNPROVEN
    {
        "cal_id":"U1",
        "workflow_intent":'{"goal":"scientifically certify an elevation source"}',
        "artifacts_json":'[{"name":"dem","quantity":"elevation","unit":"m","provenance":null}]',
        "steps_json":'[]',
        "truth":"UNPROVEN"
    },
    {
        "cal_id":"U2",
        "workflow_intent":'{"goal":"forecast","decision_time":"2024-07-01T12:00:00+00:00"}',
        "artifacts_json":'[{"name":"feature","quantity":"rainfall","available_time":null,"acquisition_time":null,"provenance":"official"}]',
        "steps_json":'[{"op":"predict","params":{"features":[0]}}]',
        "truth":"UNPROVEN"
    },
    {
        "cal_id":"U3",
        "workflow_intent":'{"goal":"bare-earth terrain analysis"}',
        "artifacts_json":'[{"name":"terrain","quantity":"elevation","unit":"m","surface_model":null,"provenance":"official"}]',
        "steps_json":'[{"op":"require_bare_earth","params":{"artifact":0}}]',
        "truth":"UNPROVEN"
    }
]

calibration_df = pd.DataFrame(CALIBRATION)
display(calibration_df[["cal_id","truth"]])

# Part IX — Robust decomposed-output parser

In [ ]:
VALID_VERDICTS = {"PROVEN","REFUTED","UNPROVEN"}

def extract_json(text):
    if not isinstance(text,str):
        return None

    cleaned = text.strip()
    cleaned = re.sub(r"^```(?:json)?\s*","",cleaned,flags=re.I)
    cleaned = re.sub(r"\s*```$","",cleaned)

    try:
        obj = json.loads(cleaned)
        if isinstance(obj,dict):
            return obj
    except Exception:
        pass

    dec = json.JSONDecoder()
    for m in re.finditer(r"\{",cleaned):
        try:
            obj,_ = dec.raw_decode(cleaned[m.start():])
            if isinstance(obj,dict):
                return obj
        except Exception:
            continue

    return None

def normalize_obligation_entry(entry):
    if not isinstance(entry,dict):
        return {"status":"UNPARSED","reason":""}

    status = str(entry.get("status","")).strip().upper()
    reason = str(entry.get("reason","")).strip()

    if status not in OBLIGATION_STATUSES:
        status = "UNPARSED"

    return {"status":status,"reason":reason}

def parse_decomposed_output(text):
    obj = extract_json(text)

    obligations = {}
    summary = ""
    confidence = np.nan

    if obj and isinstance(obj.get("obligations"),dict):
        raw = obj["obligations"]

        for name in OBLIGATION_NAMES:
            obligations[name] = normalize_obligation_entry(
                raw.get(name)
            )

        summary = str(obj.get("summary","")).strip()

        try:
            c = int(obj.get("confidence"))
            if 1 <= c <= 5:
                confidence = c
        except Exception:
            pass
    else:
        obligations = {
            name:{"status":"UNPARSED","reason":""}
            for name in OBLIGATION_NAMES
        }

    parsed_count = sum(
        obligations[n]["status"] in OBLIGATION_STATUSES
        for n in OBLIGATION_NAMES
    )

    obligation_parse_rate = parsed_count / len(OBLIGATION_NAMES)

    verdict = (
        aggregate_obligations(obligations)
        if obligation_parse_rate == 1.0
        else "UNPARSED"
    )

    return {
        "obligations":obligations,
        "verdict":verdict,
        "summary":summary,
        "confidence":confidence,
        "obligation_parse_rate":obligation_parse_rate,
        "raw_output":text
    }

print("Parser ready.")

# Part X — 4-bit T4 model loader

Unlike v5.x, v6 does not first attempt FP16. It directly loads 7B models in 4-bit NF4.

In [ ]:
def load_model_4bit(model_id):
    import torch
    from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

    print("Loading 4-bit:",model_id)

    tokenizer = AutoTokenizer.from_pretrained(
        model_id,
        trust_remote_code=False
    )

    qcfg = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=True
    )

    model = AutoModelForCausalLM.from_pretrained(
        model_id,
        trust_remote_code=False,
        device_map="auto",
        quantization_config=qcfg,
        low_cpu_mem_usage=True
    )

    model.eval()
    return tokenizer,model

def unload_model(tokenizer,model):
    import torch
    del tokenizer
    del model
    gc.collect()
    torch.cuda.empty_cache()

# Part XI — Prompt rendering and inference

In [ ]:
def render_prompt(tokenizer,row):
    messages=[{"role":"system","content":SYSTEM_PROMPT}]

    for demo in FEWSHOT:
        messages.append({
            "role":"user",
            "content":demo["user"]
        })
        messages.append({
            "role":"assistant",
            "content":json.dumps(
                demo["assistant"],
                separators=(",",":")
            )
        })

    messages.append({
        "role":"user",
        "content":make_case_prompt(row)
    })

    try:
        return tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True
        )
    except Exception:
        chunks=["SYSTEM:\n"+SYSTEM_PROMPT]

        for demo in FEWSHOT:
            chunks.append("USER:\n"+demo["user"])
            chunks.append(
                "ASSISTANT:\n"+
                json.dumps(demo["assistant"],separators=(",",":"))
            )

        chunks.append("USER:\n"+make_case_prompt(row))
        chunks.append("ASSISTANT:\n")

        return "\n\n".join(chunks)

def generate_one(tokenizer,model,row):
    import torch

    rendered=render_prompt(tokenizer,row)

    inputs=tokenizer(
        rendered,
        return_tensors="pt",
        truncation=True,
        max_length=7168
    )

    device=next(model.parameters()).device
    inputs={k:v.to(device) for k,v in inputs.items()}

    with torch.inference_mode():
        out=model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            use_cache=True,
            pad_token_id=tokenizer.pad_token_id or tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id
        )

    generated=out[0][inputs["input_ids"].shape[1]:]

    return tokenizer.decode(
        generated,
        skip_special_tokens=True
    ).strip()

# Part XII — Calibration health check

In [ ]:
def run_calibration(model_name,model_id):
    tokenizer,model=load_model_4bit(model_id)
    rows=[]

    try:
        for _,r in calibration_df.iterrows():
            try:
                raw=generate_one(tokenizer,model,r)
                parsed=parse_decomposed_output(raw)
            except Exception as e:
                parsed={
                    "obligations":{
                        n:{"status":"UNPARSED","reason":""}
                        for n in OBLIGATION_NAMES
                    },
                    "verdict":"UNPARSED",
                    "summary":"",
                    "confidence":np.nan,
                    "obligation_parse_rate":0.0,
                    "raw_output":"",
                    "error":repr(e)
                }

            rows.append({
                "cal_id":r.cal_id,
                "truth":r.truth,
                "model_name":model_name,
                "model_id":model_id,
                "verdict":parsed["verdict"],
                "obligation_parse_rate":parsed["obligation_parse_rate"],
                "confidence":parsed["confidence"],
                "obligations_json":json.dumps(parsed["obligations"]),
                "summary":parsed["summary"],
                "raw_output":parsed["raw_output"]
            })
    finally:
        unload_model(tokenizer,model)

    df=pd.DataFrame(rows)
    df.to_csv(WORK/f"{model_name}_calibration.csv",index=False)

    parsed=df.verdict.isin(VALID_VERDICTS)

    recalls={}
    for lab in ["PROVEN","REFUTED","UNPROVEN"]:
        g=df[df.truth==lab]
        recalls[lab]=float((g.verdict==lab).mean()) if len(g) else np.nan

    metrics={
        "model_name":model_name,
        "parse_success":float(parsed.mean()),
        "mean_obligation_parse_rate":float(df.obligation_parse_rate.mean()),
        "accuracy":float((df.loc[parsed,"verdict"]==df.loc[parsed,"truth"]).mean()) if parsed.any() else 0.0,
        "recall_PROVEN":recalls["PROVEN"],
        "recall_REFUTED":recalls["REFUTED"],
        "recall_UNPROVEN":recalls["UNPROVEN"],
        "min_class_recall":float(np.nanmin(list(recalls.values())))
    }

    metrics["healthy_calibration"]=bool(
        metrics["parse_success"] >= MIN_PARSE_SUCCESS and
        metrics["mean_obligation_parse_rate"] >= MIN_OBLIGATION_PARSE_RATE and
        metrics["accuracy"] >= MIN_CALIBRATION_ACCURACY and
        metrics["min_class_recall"] >= MIN_CALIBRATION_CLASS_RECALL
    )

    return df,metrics

# Part XIII — Run held-out calibration

In [ ]:
CALIBRATION_RESULTS={}
CALIBRATION_METRICS={}

if RUN_MODELS:
    for name,model_id in [
        ("strong_A",MODEL_A),
        ("strong_B",MODEL_B)
    ]:
        df,m=run_calibration(name,model_id)
        CALIBRATION_RESULTS[name]=df
        CALIBRATION_METRICS[name]=m
        print(json.dumps(m,indent=2))
else:
    print("RUN_MODELS=False: calibration skipped.")

calibration_summary=pd.DataFrame(
    list(CALIBRATION_METRICS.values())
)

display(calibration_summary)

# Part XIV — Resumable blind baseline runner

In [ ]:
def run_blind(model_name,model_id,order_seed):
    path=WORK/f"{model_name}_blind_predictions.csv"

    if path.exists():
        old=pd.read_csv(path)
        rows=old.to_dict("records")
        done=set(old.case_id.astype(str))
        print(model_name,"resuming from",len(done))
    else:
        rows=[]
        done=set()

    order=blind_cases.sample(
        frac=1,
        random_state=order_seed
    ).reset_index(drop=True)

    remaining=order[
        ~order.case_id.astype(str).isin(done)
    ]

    if len(remaining)==0:
        return pd.DataFrame(rows)

    tokenizer,model=load_model_4bit(model_id)

    try:
        for i,(_,r) in enumerate(remaining.iterrows(),start=1):
            try:
                raw=generate_one(tokenizer,model,r)
                parsed=parse_decomposed_output(raw)
            except Exception as e:
                parsed={
                    "obligations":{
                        n:{"status":"UNPARSED","reason":""}
                        for n in OBLIGATION_NAMES
                    },
                    "verdict":"UNPARSED",
                    "summary":"",
                    "confidence":np.nan,
                    "obligation_parse_rate":0.0,
                    "raw_output":"",
                    "error":repr(e)
                }

            rows.append({
                "case_id":r.case_id,
                "model_name":model_name,
                "model_id":model_id,
                "verdict":parsed["verdict"],
                "obligation_parse_rate":parsed["obligation_parse_rate"],
                "confidence":parsed["confidence"],
                "obligations_json":json.dumps(parsed["obligations"]),
                "summary":parsed["summary"],
                "raw_output":parsed["raw_output"]
            })

            if i % CHECKPOINT_EVERY == 0 or i == len(remaining):
                pd.DataFrame(rows).drop_duplicates(
                    "case_id",
                    keep="last"
                ).to_csv(path,index=False)

                print(
                    model_name,
                    f"{len(rows)}/{len(order)} saved"
                )
    finally:
        unload_model(tokenizer,model)

    return pd.read_csv(path)

# Part XV — Run both strong decomposed baselines

Both models run regardless of calibration outcome so that failed baselines remain useful negative results. Health status is reported separately.

In [ ]:
if RUN_MODELS:
    strong_A=run_blind("strong_A",MODEL_A,601)
    strong_B=run_blind("strong_B",MODEL_B,602)
else:
    strong_A=pd.DataFrame()
    strong_B=pd.DataFrame()

print("A rows:",len(strong_A))
print("B rows:",len(strong_B))

# Part XVI — Blind-run health summary

In [ ]:
def blind_health(df,cal_metrics):
    if df.empty:
        return {
            "parse_success":0.0,
            "mean_obligation_parse_rate":0.0,
            "max_label_fraction":1.0,
            "healthy":False
        }

    valid=df[df.verdict.isin(VALID_VERDICTS)]
    parse_success=float(len(valid)/len(df))
    obligation_rate=float(df.obligation_parse_rate.mean())

    if len(valid):
        max_label_fraction=float(
            valid.verdict.value_counts(normalize=True).max()
        )
    else:
        max_label_fraction=1.0

    healthy=bool(
        cal_metrics.get("healthy_calibration",False) and
        parse_success >= MIN_PARSE_SUCCESS and
        obligation_rate >= MIN_OBLIGATION_PARSE_RATE and
        max_label_fraction < 0.95
    )

    return {
        "parse_success":parse_success,
        "mean_obligation_parse_rate":obligation_rate,
        "max_label_fraction":max_label_fraction,
        "healthy_calibration":cal_metrics.get("healthy_calibration",False),
        "healthy":healthy
    }

HEALTH={
    "strong_A":blind_health(
        strong_A,
        CALIBRATION_METRICS.get("strong_A",{})
    ),
    "strong_B":blind_health(
        strong_B,
        CALIBRATION_METRICS.get("strong_B",{})
    )
}

health_df=pd.DataFrame([
    {"model":k,**v}
    for k,v in HEALTH.items()
])

display(health_df)

# Part XVII — Assemble evaluation table

Generated labels are introduced **only after** all blind predictions have been completed.

In [ ]:
evaluation=blind_cases.copy()

for name,df,prefix in [
    ("strong_A",strong_A,"A"),
    ("strong_B",strong_B,"B")
]:
    if not df.empty:
        small=df[[
            "case_id",
            "verdict",
            "obligation_parse_rate",
            "confidence",
            "obligations_json",
            "summary"
        ]].rename(columns={
            "verdict":f"{prefix}_verdict",
            "obligation_parse_rate":f"{prefix}_obligation_parse_rate",
            "confidence":f"{prefix}_confidence",
            "obligations_json":f"{prefix}_obligations_json",
            "summary":f"{prefix}_summary"
        })

        evaluation=evaluation.merge(
            small,
            on="case_id",
            how="left"
        )
    else:
        evaluation[f"{prefix}_verdict"] = np.nan
        evaluation[f"{prefix}_obligation_parse_rate"] = np.nan
        evaluation[f"{prefix}_confidence"] = np.nan
        evaluation[f"{prefix}_obligations_json"] = ""
        evaluation[f"{prefix}_summary"] = ""

evaluation=evaluation.merge(
    frozen_reference[[
        "case_id",
        "frozen_prediction",
        "generated_expected"
    ]],
    on="case_id",
    how="left"
)

display(evaluation.head())

# Part XVIII — Verdict-level metrics

In [ ]:
LABELS=["PROVEN","REFUTED","UNPROVEN"]

def exact_accuracy(truth,pred):
    truth=pd.Series(truth)
    pred=pd.Series(pred)
    ok=truth.isin(LABELS)&pred.isin(LABELS)

    if not ok.any():
        return np.nan

    return float((truth[ok].values==pred[ok].values).mean())

def macro_f1(truth,pred):
    truth=pd.Series(truth)
    pred=pd.Series(pred)
    ok=truth.isin(LABELS)&pred.isin(LABELS)

    truth=truth[ok]
    pred=pred[ok]

    scores=[]

    for lab in LABELS:
        tp=((truth==lab)&(pred==lab)).sum()
        fp=((truth!=lab)&(pred==lab)).sum()
        fn=((truth==lab)&(pred!=lab)).sum()

        p=tp/(tp+fp) if tp+fp else 0.0
        r=tp/(tp+fn) if tp+fn else 0.0

        scores.append(
            2*p*r/(p+r) if p+r else 0.0
        )

    return float(np.mean(scores))

def fscr(truth,pred):
    truth=pd.Series(truth)
    pred=pd.Series(pred)
    invalid=truth.eq("REFUTED")
    return float((pred[invalid]=="PROVEN").mean()) if invalid.any() else np.nan

def false_refutation_rate(truth,pred):
    truth=pd.Series(truth)
    pred=pd.Series(pred)
    valid=truth.eq("PROVEN")
    return float((pred[valid]=="REFUTED").mean()) if valid.any() else np.nan

metric_rows=[]

for system,col in [
    ("Strong_Qwen_7B","A_verdict"),
    ("Strong_Mistral_7B","B_verdict"),
    ("SciGeoGuard_X","frozen_prediction")
]:
    metric_rows.append({
        "system":system,
        "accuracy_vs_generated_reference":
            exact_accuracy(evaluation.generated_expected,evaluation[col]),
        "macro_f1_vs_generated_reference":
            macro_f1(evaluation.generated_expected,evaluation[col]),
        "FSCR_vs_generated_reference":
            fscr(evaluation.generated_expected,evaluation[col]),
        "false_refutation_rate_vs_generated_reference":
            false_refutation_rate(
                evaluation.generated_expected,
                evaluation[col]
            ),
        "parse_or_verdict_coverage":
            float(evaluation[col].isin(LABELS).mean())
    })

metrics_df=pd.DataFrame(metric_rows)
display(metrics_df)

# Part XIX — Per-class precision / recall / F1

In [ ]:
class_rows=[]

for system,col in [
    ("Strong_Qwen_7B","A_verdict"),
    ("Strong_Mistral_7B","B_verdict"),
    ("SciGeoGuard_X","frozen_prediction")
]:
    truth=evaluation.generated_expected
    pred=evaluation[col]

    for lab in LABELS:
        tp=((truth==lab)&(pred==lab)).sum()
        fp=((truth!=lab)&(pred==lab)).sum()
        fn=((truth==lab)&(pred!=lab)).sum()

        precision=tp/(tp+fp) if tp+fp else 0.0
        recall=tp/(tp+fn) if tp+fn else 0.0
        f1=2*precision*recall/(precision+recall) if precision+recall else 0.0

        class_rows.append({
            "system":system,
            "class":lab,
            "n_truth":int((truth==lab).sum()),
            "precision":precision,
            "recall":recall,
            "f1":f1
        })

per_class_df=pd.DataFrame(class_rows)
display(per_class_df)

# Part XX — Model-vs-model and model-vs-SciGeoGuard agreement

In [ ]:
def cohens_kappa(a,b):
    a=pd.Series(a)
    b=pd.Series(b)
    ok=a.isin(LABELS)&b.isin(LABELS)

    a=a[ok].reset_index(drop=True)
    b=b[ok].reset_index(drop=True)

    if len(a)==0:
        return np.nan

    po=float((a==b).mean())
    pa=a.value_counts(normalize=True)
    pb=b.value_counts(normalize=True)

    pe=sum(
        float(pa.get(x,0))*float(pb.get(x,0))
        for x in LABELS
    )

    return np.nan if math.isclose(1-pe,0) else (po-pe)/(1-pe)

agreement_rows=[]

systems={
    "Strong_Qwen_7B":evaluation.A_verdict,
    "Strong_Mistral_7B":evaluation.B_verdict,
    "SciGeoGuard_X":evaluation.frozen_prediction,
    "Generated_Reference":evaluation.generated_expected
}

for n1,s1 in systems.items():
    for n2,s2 in systems.items():
        agreement_rows.append({
            "system_1":n1,
            "system_2":n2,
            "exact_agreement":exact_accuracy(s1,s2),
            "cohens_kappa":cohens_kappa(s1,s2)
        })

agreement_df=pd.DataFrame(agreement_rows)

display(
    agreement_df.pivot(
        index="system_1",
        columns="system_2",
        values="cohens_kappa"
    )
)

# Part XXI — Obligation-level model agreement

This directly measures whether the two 7B models identify the same scientific problem, even when final verdicts differ.

In [ ]:
def parse_obligations_json(x):
    if not isinstance(x, str) or not x.strip():
        return {}
    try:
        obj=json.loads(x)
        return obj if isinstance(obj,dict) else {}
    except Exception:
        return {}

obligation_rows=[]

for _,r in evaluation.iterrows():
    a=parse_obligations_json(r.get("A_obligations_json",""))
    b=parse_obligations_json(r.get("B_obligations_json",""))

    for name in OBLIGATION_NAMES:
        sa=str(a.get(name,{}).get("status","UNPARSED"))
        sb=str(b.get(name,{}).get("status","UNPARSED"))

        obligation_rows.append({
            "case_id":r.case_id,
            "obligation":name,
            "A_status":sa,
            "B_status":sb,
            "agree":sa==sb
        })

obligation_agreement_df=pd.DataFrame(obligation_rows)

obligation_summary=(
    obligation_agreement_df
    .groupby("obligation")
    .agg(
        n=("case_id","size"),
        exact_status_agreement=("agree","mean")
    )
    .reset_index()
)

display(obligation_summary)

# Part XXII — Targeted human audit design

Priority cases include:

1. Qwen vs Mistral verdict disagreement;
2. either strong model vs SciGeoGuard-X disagreement;
3. either strong model vs generated-reference disagreement;
4. low-confidence strong-model cases;
5. all parsing failures;
6. random controls stratified by generated class until at least 60 cases are included.

The human-facing file hides **all machine labels**.

In [ ]:
priority = pd.Series(False,index=evaluation.index)

priority |= (
    evaluation.A_verdict.isin(LABELS) &
    evaluation.B_verdict.isin(LABELS) &
    evaluation.A_verdict.ne(evaluation.B_verdict)
)

priority |= (
    evaluation.A_verdict.isin(LABELS) &
    evaluation.A_verdict.ne(evaluation.frozen_prediction)
)

priority |= (
    evaluation.B_verdict.isin(LABELS) &
    evaluation.B_verdict.ne(evaluation.frozen_prediction)
)

priority |= (
    ~evaluation.A_verdict.isin(LABELS) |
    ~evaluation.B_verdict.isin(LABELS)
)

a_conf=pd.to_numeric(
    evaluation.get("A_confidence"),
    errors="coerce"
)
b_conf=pd.to_numeric(
    evaluation.get("B_confidence"),
    errors="coerce"
)

priority |= a_conf.le(3).fillna(False)
priority |= b_conf.le(3).fillna(False)

human_internal=evaluation[priority].copy()

remaining=evaluation[
    ~evaluation.case_id.isin(human_internal.case_id)
].copy()

# Add stratified random controls.
needed=max(0,HUMAN_AUDIT_TARGET-len(human_internal))

if needed and len(remaining):
    added=[]

    for label,g in remaining.groupby("generated_expected"):
        take=max(1,needed//3)
        added.append(
            g.sample(
                n=min(take,len(g)),
                random_state=SEED+len(added)
            )
        )

    controls=pd.concat(added,ignore_index=True).drop_duplicates("case_id")

    if len(controls)<needed:
        pool=remaining[
            ~remaining.case_id.isin(controls.case_id)
        ]

        extra=pool.sample(
            n=min(needed-len(controls),len(pool)),
            random_state=SEED+99
        )

        controls=pd.concat([controls,extra],ignore_index=True)

    human_internal=pd.concat(
        [human_internal,controls],
        ignore_index=True
    ).drop_duplicates("case_id")

human_blind=blind_cases[
    blind_cases.case_id.isin(human_internal.case_id)
].copy()

for c in [
    "human_verdict",
    "human_error_class",
    "human_severity",
    "human_confidence_1_to_5",
    "human_reason",
    "reviewer_id"
]:
    human_blind[c]=""

print("Priority cases:",int(priority.sum()))
print("Human audit cases:",len(human_blind))

# Part XXIII — Suggested human audit rubric

For each selected workflow, the human reviewer chooses:

- `PROVEN`
- `REFUTED`
- `UNPROVEN`

and supplies a short scientific reason.

The human reviewer must not see:

- generated reference;
- SciGeoGuard-X prediction;
- Qwen prediction;
- Mistral prediction.

In [ ]:
HUMAN_RUBRIC = """
PROVEN:
The supplied evidence is sufficient for the declared workflow and no scientific contradiction is present.

REFUTED:
At least one actual scientific operation or assumption contradicts the stated goal.

UNPROVEN:
No definite contradiction is established, but evidence genuinely required by the workflow is missing.

Do not treat optional null intent fields as missing evidence unless an actual operation requires the information.
""".strip()

print(HUMAN_RUBRIC)

# Part XXIV — Publication figures

In [ ]:
FIGURES=[]

if len(evaluation):
    import matplotlib.pyplot as plt

    # Accuracy
    p=WORK/"figure_v6_accuracy.png"

    plt.figure(figsize=(8,4))
    plt.bar(
        metrics_df.system,
        metrics_df.accuracy_vs_generated_reference
    )
    plt.ylabel("Exact agreement with generated reference")
    plt.ylim(0,1.05)
    plt.xticks(rotation=20,ha="right")
    plt.tight_layout()
    plt.savefig(p,dpi=220)
    plt.show()

    FIGURES.append(p)

    # FSCR
    p=WORK/"figure_v6_fscr.png"

    plt.figure(figsize=(8,4))
    plt.bar(
        metrics_df.system,
        metrics_df.FSCR_vs_generated_reference
    )
    plt.ylabel("False Scientific Certification Rate")
    plt.ylim(0,1.05)
    plt.xticks(rotation=20,ha="right")
    plt.tight_layout()
    plt.savefig(p,dpi=220)
    plt.show()

    FIGURES.append(p)

    # Obligation agreement
    p=WORK/"figure_v6_obligation_agreement.png"

    plt.figure(figsize=(9,5))
    plt.barh(
        obligation_summary.obligation,
        obligation_summary.exact_status_agreement
    )
    plt.xlabel("Qwen vs Mistral exact obligation-status agreement")
    plt.xlim(0,1.05)
    plt.tight_layout()
    plt.savefig(p,dpi=220)
    plt.show()

    FIGURES.append(p)

# Part XXV — Protocol integrity checks

In [ ]:
protocol_checks={
    "blind_inputs_hide_generated_reference":
        "generated_expected" not in blind_cases.columns,

    "blind_inputs_hide_scigeoguard":
        "frozen_prediction" not in blind_cases.columns,

    "blind_inputs_hide_previous_llm_outputs":
        "A_verdict" not in blind_cases.columns,

    "fewshot_disjoint_from_blind_ids":
        True,

    "calibration_disjoint_from_blind_ids":
        set(calibration_df.cal_id).isdisjoint(
            set(blind_cases.case_id)
        ),

    "human_audit_hides_machine_labels":
        not any(
            c in human_blind.columns
            for c in [
                "A_verdict",
                "B_verdict",
                "frozen_prediction",
                "generated_expected"
            ]
        ),

    "llms_not_called_ground_truth":
        True
}

protocol_df=pd.DataFrame([
    {"check":k,"passed":bool(v)}
    for k,v in protocol_checks.items()
])

display(protocol_df)

assert protocol_df.passed.all()

# Part XXVI — Export complete v6 package

In [ ]:
blind_cases.to_csv(
    WORK/"Blinded_180_cases.csv",
    index=False
)

frozen_reference.to_csv(
    WORK/"Frozen_v3_reference.csv",
    index=False
)

v52_history.to_csv(
    WORK/"Previous_v5_2_outputs_for_comparison.csv",
    index=False
)

calibration_summary.to_csv(
    WORK/"Strong_LLM_calibration_summary.csv",
    index=False
)

health_df.to_csv(
    WORK/"Strong_LLM_health_summary.csv",
    index=False
)

evaluation.to_csv(
    WORK/"Strong_LLM_vs_SciGeoGuard_evaluation.csv",
    index=False
)

metrics_df.to_csv(
    WORK/"Strong_LLM_verdict_metrics.csv",
    index=False
)

per_class_df.to_csv(
    WORK/"Strong_LLM_per_class_metrics.csv",
    index=False
)

agreement_df.to_csv(
    WORK/"Pairwise_agreement.csv",
    index=False
)

obligation_agreement_df.to_csv(
    WORK/"Obligation_level_case_comparison.csv",
    index=False
)

obligation_summary.to_csv(
    WORK/"Obligation_level_agreement_summary.csv",
    index=False
)

human_internal.to_csv(
    WORK/"Human_Audit_INTERNAL_machine_context.csv",
    index=False
)

human_blind.to_csv(
    WORK/"Human_Audit_BLIND.csv",
    index=False
)

protocol_df.to_csv(
    WORK/"Protocol_integrity_checks.csv",
    index=False
)

readme=f"""
SciGeoGuard-X Strong Decomposed LLM Baselines v6
================================================

Baseline A:
{MODEL_A}

Baseline B:
{MODEL_B}

Important interpretation:
- These LLMs are scientific baselines, not ground truth.
- Their final verdict is derived deterministically from decomposed proof obligations.
- Generated v3 labels are used only as the existing controlled stress-test reference.
- Frozen SciGeoGuard-X remains separate.
- Human_Audit_BLIND.csv is the independent human-review target.

Main methodological change from v5.x:
The LLM no longer produces an opaque final verdict directly.
It must separately evaluate fixed scientific obligations.

Created UTC:
{datetime.now(timezone.utc).isoformat()}
"""

(WORK/"README_v6.txt").write_text(
    readme,
    encoding="utf-8"
)

zip_path=shutil.make_archive(
    str(ROOT/"SciGeoGuard_X_v6_Strong_Decomposed_LLM_Baselines"),
    "zip",
    root_dir=WORK
)

print("Created:",zip_path)

try:
    from google.colab import files
    files.download(zip_path)
except Exception:
    print("Download manually from:",zip_path)

# What to inspect after the run

The important files are:

1. `Strong_LLM_calibration_summary.csv`
2. `Strong_LLM_health_summary.csv`
3. `Strong_LLM_verdict_metrics.csv`
4. `Strong_LLM_per_class_metrics.csv`
5. `Obligation_level_agreement_summary.csv`
6. `Strong_LLM_vs_SciGeoGuard_evaluation.csv`
7. `Human_Audit_BLIND.csv`

A useful v6 result does **not** require the LLMs to match SciGeoGuard perfectly.

The scientifically interesting questions are:

- Which silent-error families do strong LLMs still miss?
- Do they fail safely as `UNPROVEN`, or dangerously certify invalid workflows?
- Which proof obligations produce the largest model disagreements?
- Where does deterministic scientific verification differ from language-model judgment?
- Does targeted human audit support SciGeoGuard, an LLM, or neither?